# Lesson 10 - AI Agents in Production

In this lesson you will learn **production patterns** for AI agents using the Microsoft Agent Framework (Python). We cover:

- **Observability** — додавання відстеження часу та логування взаємодій агента
- **Evaluation** — використання агента-оцінювача для оцінки якості відповідей
- **Cost management** — стратегії оптимізації токенів та вибору моделей

The scenario is a **travel agent** that helps users plan trips, with monitoring and evaluation layered on top.


## Налаштування


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Виробничі аспекти

Перенесення AI-агентів з прототипів у виробництво вимагає ретельної уваги до трьох складових:

1. **Спостережуваність** — Потрібен огляд того, що робить агент, скільки це займає часу та які інструменти він викликає. Без трасування та логування відлагодження проблем у виробництві практично неможливе.

2. **Оцінка** — Автоматизовані перевірки якості забезпечують, що відповіді агента залишаються точними, повними та корисними з часом. Агент-оцінювач може ставити оцінки відповідям відповідно до визначених критеріїв.

3. **Управління витратами** — Використання токенів безпосередньо впливає на вартість. Стратегії на кшталт оптимізації запитів, вибору моделі та кешування допомагають контролювати витрати без втрати якості.


## Побудова спостережуваного агента

Ми визначаємо інструменти для подорожей і обгортаємо виклик агента з вимірюванням часу, щоб ми могли відстежувати затримку. У виробничому середовищі ви б інтегрувалися з OpenTelemetry або подібною системою трасування.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Шаблони оцінювання

Поширеним виробничим шаблоном є використання другого агента як **оцінювача**. Оцінювач оцінює відповідь основного агента за заздалегідь визначеними критеріями, такими як повнота, точність і корисність.

Це дає змогу:
- Автоматизувати контроль якості перед тим, як відповіді доходять до користувачів
- Виявляти регресії при зміні запитів або моделей
- Постійно моніторити продуктивність агента з часом


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Стратегії управління витратами

Контроль витрат є критично важливим для виробничих AI-агентів. Ось основні стратегії:

| Стратегія | Опис |
|---|---|
| **Оптимізація промптів** | Тримайте системні інструкції короткими. Видаляйте надлишковий контекст для зменшення кількості токенів введення. |
| **Вибір моделі** | Використовуйте менші, дешевші моделі (наприклад, GPT-4o-mini) для простих завдань, таких як класифікація або вилучення, і залишайте більші моделі для складного логічного мислення. |
| **Кешування** | Кешуйте результати інструментів і часті запити, щоб уникнути дублюючих викликів API. |
| **Бюджети токенів** | Встановлюйте обмеження `max_tokens`, щоб запобігти несподівано довгим відповідям. |
| **Пакетна обробка** | Об’єднуйте кілька запитів користувача в один виклик API, якщо це можливо. |

На практиці добре працює багаторівневий підхід: спрямовуйте прості запити до швидкої, недорогової моделі і ескалюйте складні запити лише до більш потужної.


## Підсумок

У цьому уроці ви дізналися, як:

1. **Додати спостереження** за взаємодіями агента з вимірюванням часу та журналюванням, закладаючи основу для трасування та моніторингу.
2. **Автоматично оцінювати відповіді агента** за допомогою агента-оцінювача, який оцінює повноту, точність і корисність.
3. **Керувати витратами** через оптимізацію підказок, вибір моделей, кешування та бюджети токенів.

Ці виробничі патерни допомагають гарантувати, що ваші AI-агенти є надійними, вимірюваними та економічними у масштабі.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
